In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, Dropout
from tensorflow.keras.callbacks import EarlyStopping

ticker = "GOOGL"
df = yf.download(ticker, start="2018-01-01", end="2024-01-01")
print(f"Dataset Shape: {df.shape}")
print(f"Date Range: {df.index.min().date()} to {df.index.max().date()}")
df.head()

In [ ]:
print(df.info())
print(df.describe())
print(df.isnull().sum())

In [ ]:
plt.figure(figsize=(16,6))
plt.plot(df.index, df['Close'])
plt.show()

In [ ]:
data = df[['Close']].values
scaler = MinMaxScaler(feature_range=(0,1))
data_scaled = scaler.fit_transform(data)

In [ ]:
def create_sequences(data, time_steps=60):
    X, y = [], []
    for i in range(time_steps, len(data)):
        X.append(data[i-time_steps:i,0])
        y.append(data[i,0])
    return np.array(X), np.array(y)

TIME_STEPS = 60
train_size = int(len(data_scaled)*0.8)
train_data = data_scaled[:train_size]
test_data = data_scaled[train_size-TIME_STEPS:]

X_train, y_train = create_sequences(train_data, TIME_STEPS)
X_test, y_test = create_sequences(test_data, TIME_STEPS)

X_train = X_train.reshape((X_train.shape[0], X_train.shape[1],1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1],1))

In [ ]:
model = Sequential([
    SimpleRNN(64, return_sequences=True, input_shape=(TIME_STEPS,1)),
    Dropout(0.2),
    SimpleRNN(64),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [ ]:
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1)

In [ ]:
y_pred = scaler.inverse_transform(model.predict(X_test))
y_actual = scaler.inverse_transform(y_test.reshape(-1,1))

print("MSE:", mean_squared_error(y_actual,y_pred))